# Pharmakaksha · BP701T · Unit III · Correlation and Regression
**Learn · Grow · Achieve**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit3_Correlation_and_Regression.ipynb) &nbsp; [Open on learn.pharmakaksha.com](https://learn.pharmakaksha.com/notebooks/index.html?path=BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit3_Correlation_and_Regression.ipynb)

Companion notebook for BP701T Unit III study notes, slides and video. Every worked example from the unit is here, in order, followed by lab exercises and solutions. Dataset: `tablet_dissolution.csv`.

> **How to use this notebook:** BP701T is a hand-calculation course. Solve each example with your calculator first, using the study notes, then run the cell to check your answer. In the exam you will not have Python.

## Using learn.pharmakaksha.com
- **First run is slow:** Python loads in your browser, which takes about 10 seconds the first time.
- **Kernel indicator:** the circle at the top right is filled while a cell runs and empty when it is idle.
- **Restart:** Kernel → Restart Kernel if something gets stuck, then run the cells again from the top.
- **Saving:** your changes are saved only in this browser. Download the notebook (File → Download) to keep a copy.
- **Start over:** delete your copy in the file browser to get a fresh notebook from the site.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("Ready")

In [ ]:
import os
RAW = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/"
def load(name):
    """Read a practice CSV from this folder, or from GitHub when running in Colab."""
    return pd.read_csv(name) if os.path.exists(name) else pd.read_csv(RAW + name)

## 3.2 Pearson's correlation coefficient
Dose (mg) and fall in diastolic BP (mmHg) in 6 patients. Hand answers: Σdxdy = 340, Σdx² = 437.5, Σdy² = 269.33, r = 0.990.

In [ ]:
x = np.array([5, 10, 15, 20, 25, 30]); y = np.array([8, 12, 15, 21, 22, 28])
dx, dy = x - x.mean(), y - y.mean()
work = pd.DataFrame({"x": x, "y": y, "dx": dx, "dy": dy.round(2), "dx*dy": (dx*dy).round(2), "dx²": dx**2, "dy²": (dy**2).round(2)})
print(work, "\n")
Sxy, Sxx, Syy = (dx*dy).sum(), (dx**2).sum(), (dy**2).sum()
r = Sxy / np.sqrt(Sxx * Syy)
print(f"Sxy {Sxy:.2f}  Sxx {Sxx:.2f}  Syy {Syy:.2f}  r = {r:.3f}  r² = {r*r:.3f}")
print(stats.pearsonr(x, y))
t_r = r * np.sqrt(len(x) - 2) / np.sqrt(1 - r*r)
print(f"t = {t_r:.2f} vs t(0.05, df 4) = {stats.t.ppf(0.975, 4):.3f}")

In [ ]:
plt.scatter(x, y, color="#1F8A70", s=60)
b, a = np.polyfit(x, y, 1)
plt.plot(x, a + b*x, color="#E07A2E", label=f"y = {a:.2f} + {b:.3f}x")
plt.xlabel("Dose (mg)"); plt.ylabel("Fall in DBP (mmHg)"); plt.legend(); plt.title("Scatter diagram: strong positive correlation"); plt.show()

## 3.3 Spearman's rank correlation
Two panelists rank 8 formulations for taste. Hand answer: Σd² = 8, ρ = 0.905.

In [ ]:
r1 = np.arange(1, 9); r2 = np.array([2, 1, 4, 3, 6, 5, 8, 7])
d = r1 - r2; n = len(d)
print("Σd² =", (d**2).sum(), "| rho =", round(1 - 6*(d**2).sum() / (n*(n**2 - 1)), 3))
print(stats.spearmanr(r1, r2))
# With tied scores, give average ranks
s1 = np.array([7, 5, 8, 6, 9, 4, 6, 3]); s2 = np.array([6, 6, 9, 5, 8, 3, 7, 4])
print("ranks:", stats.rankdata(s1), stats.rankdata(s2))
print("rho with ties (scipy, exact correction):", round(stats.spearmanr(s1, s2).statistic, 3))

## 3.4 Multiple and partial correlation
r₁₂ = 0.80, r₁₃ = 0.60, r₂₃ = 0.50. Hand answers: R₁.₂₃ = 0.833, r₁₂.₃ = 0.722.

In [ ]:
r12, r13, r23 = 0.80, 0.60, 0.50
R = np.sqrt((r12**2 + r13**2 - 2*r12*r13*r23) / (1 - r23**2))
partial = (r12 - r13*r23) / np.sqrt((1 - r13**2) * (1 - r23**2))
print(f"R1.23 = {R:.3f} | r12.3 = {partial:.3f}")

## 3.5–3.6 Least-squares regression and regression coefficients
Hand answers: b_yx = 0.777, a = 4.07, ŷ = 4.07 + 0.777x, b_xy = 1.262, r = √(b_yx × b_xy) = 0.990.

In [ ]:
byx = Sxy / Sxx; a = y.mean() - byx * x.mean(); bxy = Sxy / Syy
print(f"y on x: y = {a:.3f} + {byx:.4f} x | x on y slope bxy = {bxy:.4f}")
print("r from the two coefficients =", round(np.sqrt(byx * bxy), 4))
print("Predicted fall at 22 mg =", round(a + byx*22, 2), "mmHg")
res = stats.linregress(x, y)
print(res)

## 3.7 Multiple linear regression
`tablet_dissolution.csv`: % dissolved at 30 min for 12 batches with different binder % and hardness.

In [ ]:
import statsmodels.formula.api as smf
td = load("tablet_dissolution.csv")
print(td.head())
model = smf.ols("dissolution_30min_pct ~ binder_pct + hardness_kg_cm2", data=td).fit()
print(model.params.round(3))
print("R² =", round(model.rsquared, 3), "| adjusted R² =", round(model.rsquared_adj, 3))
print("Binder only R² =", round(smf.ols("dissolution_30min_pct ~ binder_pct", data=td).fit().rsquared, 3))

## 3.8 Over- and underfitting
A straight line on curved data **underfits**; a high-degree polynomial through every point **overfits** and predicts new doses badly.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
xs = np.linspace(5, 30, 200)
for axi, deg, lab in zip(ax, (0, 1, 5), ("Underfit: degree 0", "Good fit: degree 1", "Overfit: degree 5")):
    coef = np.polyfit(x, y, deg)
    axi.scatter(x, y, color="#1F8A70"); axi.plot(xs, np.polyval(coef, xs), color="#E07A2E"); axi.set_title(lab)
plt.tight_layout(); plt.show()

## Worked example · UV calibration curve (paracetamol, 243 nm)
Notes, worked pharma example. Hand answers: A = 0.0017 + 0.06509c, r = 0.99995, unknown = 6.89 µg/mL.

In [ ]:
conc = np.array([2, 4, 6, 8, 10, 12]); absb = np.array([0.132, 0.261, 0.395, 0.519, 0.655, 0.782])
cal = stats.linregress(conc, absb)
print(f"A = {cal.slope:.5f} c + {cal.intercept:.5f} | r = {cal.rvalue:.5f} | r² = {cal.rvalue**2:.5f}")
print("Unknown concentration =", round((0.450 - cal.intercept) / cal.slope, 2), "µg/mL")

## Lab exercises
1. Find r between age (30, 40, 50, 60, 70) and creatinine clearance (110, 98, 85, 70, 62) mL/min, and test it.
2. Fit the regression line of clearance on age and predict clearance at 55 years.
3. Two examiners rank 6 students: (1,2,3,4,5,6) and (2,1,3,6,4,5). Find Spearman's ρ.
4. If b_yx = −0.6 and b_xy = −1.2, find r.
5. r₁₂ = 0.7, r₁₃ = 0.5, r₂₃ = 0.4. Find the partial correlation r₁₂.₃.

### Solutions

In [ ]:
age = np.array([30, 40, 50, 60, 70]); crcl = np.array([110, 98, 85, 70, 62])
r1_ = stats.pearsonr(age, crcl); print("1:", round(r1_.statistic, 4), "p =", round(r1_.pvalue, 4))
l = stats.linregress(age, crcl); print("2:", round(l.intercept, 2), round(l.slope, 3), "->", round(l.intercept + l.slope*55, 1))
print("3:", stats.spearmanr([1,2,3,4,5,6], [2,1,3,6,4,5]).statistic.round(3))
print("4:", -np.sqrt(0.6*1.2).round(3), "(sign follows the coefficients)")
print("5:", round((0.7 - 0.5*0.4) / np.sqrt((1 - 0.25) * (1 - 0.16)), 3))